In [2]:

import random
import numpy as np
import matplotlib.pyplot as plt

# --- CONFIGURATION & PARAMETERS ---
PHASES = [
    'North-South Straight',
    'East-West Straight',
    'North-South Left',
    'East-West Left'
]

NUM_PHASES = len(PHASES)
MIN_GREEN, MAX_GREEN, YELLOW_TIME = 15, 60, 4

TRAFFIC_DEMAND = [0.45, 0.35, 0.15, 0.20]  # vehicles/second
ROAD_CAPACITY = [0.90, 0.85, 0.50, 0.60]   # vehicles/second

# PSO parameters
NUM_PARTICLES = 40
ITERATIONS = 50
W = 0.7                 # Inertia weight
C1 = 1.5                # Cognitive coefficient
C2 = 1.5                # Social coefficient
V_MAX = 10              # Maximum velocity magnitude

random.seed(42)
np.random.seed(42)


# --- TRAFFIC SIMULATION & FITNESS ---
def evaluate_fitness(chromosome):
    cycle_length = sum(chromosome) + NUM_PHASES * YELLOW_TIME

    total_delay = 0.0
    total_vehicles = 0.0

    max_queues = [0.0] * NUM_PHASES
    current_queues = [0.0] * NUM_PHASES

    # Generate signal timeline
    timeline = []
    for phase_idx, green_time in enumerate(chromosome):
        timeline.extend(
            [phase_idx] * int(green_time)
            + [-1] * YELLOW_TIME
        )

    # Second-by-second simulation
    for second in range(cycle_length):
        active_phase = timeline[second]

        for i in range(NUM_PHASES):
            current_queues[i] += TRAFFIC_DEMAND[i]

            if active_phase == i and current_queues[i] > 0:
                discharge = min(
                    ROAD_CAPACITY[i],
                    current_queues[i]
                )
                current_queues[i] -= discharge
                total_vehicles += discharge

            max_queues[i] = max(
                max_queues[i], current_queues[i]
            )

        total_delay += sum(current_queues)

    avg_delay = (
        total_delay / total_vehicles
        if total_vehicles > 0 else 999.0
    )

    # PSO minimizes this objective
    return avg_delay, max_queues


# --- PARTICLE SWARM OPTIMIZATION ---
print("Running Particle Swarm Optimization...")

# Each particle represents four green-time allocations
positions = np.random.uniform(
    MIN_GREEN, MAX_GREEN,
    (NUM_PARTICLES, NUM_PHASES)
)

velocities = np.random.uniform(
    -5, 5,
    (NUM_PARTICLES, NUM_PHASES)
)

personal_best_positions = positions.copy()
personal_best_scores = np.full(NUM_PARTICLES, np.inf)

global_best_position = None
global_best_score = np.inf
global_best_queues = None

history_best = []
history_avg = []

for iteration in range(ITERATIONS):

    # Evaluate every particle
    scores = []

    for i in range(NUM_PARTICLES):
        # Convert continuous position to integer green times
        positions[i] = np.clip(
            positions[i], MIN_GREEN, MAX_GREEN
        )
        green_times = np.rint(positions[i]).astype(int)

        delay, queues = evaluate_fitness(green_times)
        scores.append(delay)

        # Update personal best
        if delay < personal_best_scores[i]:
            personal_best_scores[i] = delay
            personal_best_positions[i] = positions[i].copy()

        # Update global best
        if delay < global_best_score:
            global_best_score = delay
            global_best_position = positions[i].copy()
            global_best_queues = queues.copy()

    history_best.append(global_best_score)
    history_avg.append(np.mean(scores))

    # Update particle velocities and positions
    for i in range(NUM_PARTICLES):
        r1 = np.random.random(NUM_PHASES)
        r2 = np.random.random(NUM_PHASES)

        cognitive = C1 * r1 * (
            personal_best_positions[i] - positions[i]
        )

        social = C2 * r2 * (
            global_best_position - positions[i]
        )

        velocities[i] = (
            W * velocities[i] + cognitive + social
        )

        velocities[i] = np.clip(
            velocities[i], -V_MAX, V_MAX
        )

        positions[i] += velocities[i]

        # Enforce green-time limits
        positions[i] = np.clip(
            positions[i], MIN_GREEN, MAX_GREEN
        )

    print(
        f"Iteration {iteration + 1:02d}/{ITERATIONS} | "
        f"Best Delay: {global_best_score:.2f}s"
    )


# --- FINAL RESULTS ---
best_chrom = np.rint(global_best_position).astype(int)
final_delay, best_queues = evaluate_fitness(best_chrom)

print("\nOptimization Complete!")
print(f"Best Average Delay: {final_delay:.2f}s")
print(
    "Total Cycle Time:",
    sum(best_chrom) + NUM_PHASES * YELLOW_TIME,
    "seconds"
)

for i, green in enumerate(best_chrom):
    print(
        f"{PHASES[i]}: {green}s Green Allocation "
        f"(Max Queue: {best_queues[i]:.1f} vehicles)"
    )

"""
# --- PERFORMANCE PLOTS ---
plt.figure(figsize=(10, 5))
plt.plot(
    range(1, ITERATIONS + 1),
    history_best,
    label="Global Best Delay"
)
plt.plot(
    range(1, ITERATIONS + 1),
    history_avg,
    label="Mean Particle Delay",
    linestyle="--"
)

plt.xlabel("Iteration")
plt.ylabel("Average Delay (seconds)")
plt.title("PSO Traffic Signal Optimization")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
"""

Running Particle Swarm Optimization...
Iteration 01/50 | Best Delay: 72.50s
Iteration 02/50 | Best Delay: 68.11s
Iteration 03/50 | Best Delay: 62.76s
Iteration 04/50 | Best Delay: 57.45s
Iteration 05/50 | Best Delay: 55.40s
Iteration 06/50 | Best Delay: 55.40s
Iteration 07/50 | Best Delay: 55.40s
Iteration 08/50 | Best Delay: 55.40s
Iteration 09/50 | Best Delay: 55.40s
Iteration 10/50 | Best Delay: 55.40s
Iteration 11/50 | Best Delay: 55.40s
Iteration 12/50 | Best Delay: 55.40s
Iteration 13/50 | Best Delay: 55.40s
Iteration 14/50 | Best Delay: 55.40s
Iteration 15/50 | Best Delay: 55.40s
Iteration 16/50 | Best Delay: 55.40s
Iteration 17/50 | Best Delay: 55.40s
Iteration 18/50 | Best Delay: 55.40s
Iteration 19/50 | Best Delay: 55.40s
Iteration 20/50 | Best Delay: 55.40s
Iteration 21/50 | Best Delay: 55.40s
Iteration 22/50 | Best Delay: 55.40s
Iteration 23/50 | Best Delay: 55.40s
Iteration 24/50 | Best Delay: 55.40s
Iteration 25/50 | Best Delay: 55.40s
Iteration 26/50 | Best Delay: 55.40s

'\n# --- PERFORMANCE PLOTS ---\nplt.figure(figsize=(10, 5))\nplt.plot(\n    range(1, ITERATIONS + 1),\n    history_best,\n    label="Global Best Delay"\n)\nplt.plot(\n    range(1, ITERATIONS + 1),\n    history_avg,\n    label="Mean Particle Delay",\n    linestyle="--"\n)\n\nplt.xlabel("Iteration")\nplt.ylabel("Average Delay (seconds)")\nplt.title("PSO Traffic Signal Optimization")\nplt.legend()\nplt.grid(True)\nplt.tight_layout()\nplt.show()\n'